# DataSens — Fine-tuning sentiment FR sur Google Colab

Notebook minimal pour démontrer entraînement + évaluation + généralisation.

**Étapes**
1. Installer les dépendances (Cellule 1)
2. Uploader le zip `colab_bundle.zip` contenant `train.parquet`, `val.parquet`, `test.parquet` (Cellule 2)
3. Charger les données et préparer (Cellule 3)
4. Fine-tuner `sentiment_fr` avec class_weight, logs TensorBoard (Cellule 4)
5. Évaluer + tester sur phrases manuelles + télécharger le modèle (Cellule 5)
6. **Diagnostic pédagogique** : courbes loss/F1 + verdict apprend/overfit/généralise (Cellule 6)
7. Visualiser les courbes loss/F1 dans TensorBoard inline (Cellule 7, optionnelle)

**Comment lire les résultats** (Cellule 6 fait le verdict automatiquement) :
- *« Ça apprend »* = train loss qui descend nettement (≥ −10 %)
- *« Pas d'overfitting »* = eval loss qui descend ou stagne, ne remonte pas en fin
- *« Ça généralise »* = écart F1 macro val ↔ test < 0,05 (le modèle marche aussi bien sur des données jamais vues)
- *« Cible projet »* = F1 macro test ≥ 0,55 et F1 par classe ≥ 0,20

**Pré-requis local (déjà fait par `scripts/create_ia_copy.py` patché)**
- splits propres : pas de fuite train/test, pas de doublons, pas de textes <30 chars
- split temporel : test = données plus récentes que train

**Tracking** : `report_to='tensorboard'` — les events sont écrits dans `<OUT_DIR>/runs/`
et inclus dans le zip téléchargé en fin de notebook (visualisables localement via
`tensorboard --logdir <OUT_DIR>/runs` après réinjection).

**Recommandé** : Runtime → Modifier le type d'exécution → GPU (T4 gratuit suffit).

In [ ]:
# Cellule 1 — Dépendances
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 evaluate==0.4.3 scikit-learn==1.5.2 pyarrow==17.0.0
import torch, transformers, sklearn, pandas as pd
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)
print('sklearn', sklearn.__version__)

In [ ]:
# Cellule 2 — Upload du bundle (train/val/test.parquet en zip)
from google.colab import files
import zipfile, os

uploaded = files.upload()  # selectionner colab_bundle.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('data')
print('Fichiers extraits dans ./data :', os.listdir('data'))

In [ ]:
# Cellule 3 — Chargement, mapping label, statistiques
import pandas as pd, numpy as np
from pathlib import Path

DATA = Path('data')
train = pd.read_parquet(DATA/'train.parquet')
val   = pd.read_parquet(DATA/'val.parquet')
test  = pd.read_parquet(DATA/'test.parquet')

LABEL2ID = {'negatif': 0, 'neutre': 1, 'positif': 2}
ID2LABEL = {v:k for k,v in LABEL2ID.items()}

def normalize(label):
    s = str(label or '').strip().lower()
    if s.startswith('neg'): return 'negatif'
    if s.startswith('pos'): return 'positif'
    return 'neutre'

def prepare(df):
    title = df['title'].fillna('').astype(str) if 'title' in df.columns else ''
    content = df['content'].fillna('').astype(str) if 'content' in df.columns else df.get('text', pd.Series(['']*len(df))).fillna('').astype(str)
    text = (title + ' ' + content).str.strip().str.slice(0, 1024)
    label = df['sentiment'].apply(normalize).map(LABEL2ID)
    out = pd.DataFrame({'text': text, 'label': label}).dropna()
    out['label'] = out['label'].astype(int)
    return out

train_df = prepare(train)
val_df   = prepare(val)
test_df  = prepare(test)

print('train', len(train_df), '| val', len(val_df), '| test', len(test_df))
print('train distrib :', train_df['label'].map(ID2LABEL).value_counts().to_dict())

In [ ]:
# Cellule 4 — Fine-tuning sentiment_fr (3 epochs, class_weight balanced, GPU si dispo)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import torch

MODEL_NAME = 'ac0hik/Sentiment_Analysis_French'
MAX_LEN    = 256
EPOCHS     = 3
BATCH      = 16
OUT_DIR    = 'models/sentiment_fr-finetuned-colab'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

def to_hf(df):
    ds = Dataset.from_pandas(df.reset_index(drop=True))
    ds = ds.map(lambda b: tokenizer(b['text'], truncation=True, max_length=MAX_LEN, padding='max_length'), batched=True, remove_columns=['text'])
    ds.set_format('torch', columns=['input_ids','attention_mask','label'])
    return ds

train_ds = to_hf(train_df)
val_ds   = to_hf(val_df)
test_ds  = to_hf(test_df)

labels_arr = train_df['label'].to_numpy()
weights = compute_class_weight('balanced', classes=np.array([0,1,2]), y=labels_arr)
weights_t = torch.tensor(weights, dtype=torch.float)
print('class_weight :', dict(zip(['negatif','neutre','positif'], weights.round(3))))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        w = weights_t.to(logits.device)
        loss = torch.nn.CrossEntropyLoss(weight=w)(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.argmax(axis=-1)
    f1_pc = f1_score(labels, preds, average=None, labels=[0,1,2])
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
        'f1_neg': float(f1_pc[0]), 'f1_neu': float(f1_pc[1]), 'f1_pos': float(f1_pc[2]),
    }

args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH*2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_dir=f'{OUT_DIR}/runs',
    report_to='tensorboard',
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, padding=True),
    compute_metrics=metrics,
)
trainer.train()
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print('Modèle sauvegardé dans', OUT_DIR)

In [ ]:
# Cellule 5 — Évaluation test + 5 phrases manuelles + zip téléchargeable
from transformers import pipeline
import shutil, json

test_metrics = trainer.evaluate(eval_dataset=test_ds)
print('--- métriques sur test (jamais vu pendant training) ---')
for k,v in test_metrics.items():
    if isinstance(v,(int,float)): print(f'{k}: {v:.4f}')

device = 0 if torch.cuda.is_available() else -1
pipe = pipeline('text-classification', model=OUT_DIR, tokenizer=OUT_DIR, device=device)

phrases = [
    "Le marché affiche une hausse spectaculaire et les investisseurs sont optimistes.",
    "Forte chute des indices, les analystes sont inquiets.",
    "La banque centrale maintient ses taux à leur niveau actuel.",
    "Le service client est catastrophique, je ne reviendrai pas.",
    "Excellente expérience, je recommande vivement.",
]
print('\n--- généralisation hors dataset ---')
for p in phrases:
    r = pipe(p[:256], truncation=True, max_length=256)[0]
    print(f'[{r["label"]:8s} | {r["score"]:.2%}]  {p}')

with open(f'{OUT_DIR}/test_metrics.json','w') as f:
    json.dump({k:float(v) for k,v in test_metrics.items() if isinstance(v,(int,float))}, f, indent=2)

shutil.make_archive('sentiment_fr_finetuned_colab', 'zip', OUT_DIR)
from google.colab import files as gfiles
gfiles.download('sentiment_fr_finetuned_colab.zip')

In [ ]:
# Cellule 6 — Diagnostic pedagogique : ca apprend ? ca generalise ?
# A executer juste apres la Cellule 5. Trace les courbes et donne un verdict explicite.
import matplotlib.pyplot as plt

hist = trainer.state.log_history
train_loss = [(h['epoch'], h['loss']) for h in hist if 'loss' in h and 'eval_loss' not in h]
eval_log   = [(h['epoch'], h['eval_loss'], h['eval_f1_macro']) for h in hist if 'eval_loss' in h]

if not train_loss or not eval_log:
    print("Pas assez d'historique. Relance la Cellule 4 d'abord.")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ep_t, loss_t = zip(*train_loss)
    ep_e, loss_e, f1_e = zip(*eval_log)

    ax1.plot(ep_t, loss_t, 'o-', alpha=0.6, label='Train loss')
    ax1.plot(ep_e, loss_e, 's-', linewidth=2, label='Eval (val) loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title('Loss : ca descend = ca apprend')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(ep_e, f1_e, 's-', color='green', linewidth=2, label='F1 macro (val)')
    ax2.axhline(0.55, color='red', linestyle='--', alpha=0.5, label='Cible projet 0.55')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1 macro')
    ax2.set_title('F1 macro validation : ca monte = ca apprend')
    ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

# Comparaison val (vue pendant training) vs test (jamais vu) = generalisation
val_metrics  = trainer.evaluate(eval_dataset=val_ds)
print('\n=== Validation (vue pendant training) ===')
print(f"  loss    : {val_metrics['eval_loss']:.4f}")
print(f"  accuracy: {val_metrics['eval_accuracy']:.4f}")
print(f"  f1_macro: {val_metrics['eval_f1_macro']:.4f}")

print('\n=== Test (jamais vu pendant training) ===')
print(f"  loss    : {test_metrics['eval_loss']:.4f}")
print(f"  accuracy: {test_metrics['eval_accuracy']:.4f}")
print(f"  f1_macro: {test_metrics['eval_f1_macro']:.4f}")

# Verdict explicite
gap_f1 = val_metrics['eval_f1_macro'] - test_metrics['eval_f1_macro']
print('\n=== VERDICT ===')

if loss_t[-1] < loss_t[0] * 0.9:
    print(f"  [OK] Apprentissage   : train loss {loss_t[0]:.3f} -> {loss_t[-1]:.3f} (-{(1-loss_t[-1]/loss_t[0])*100:.0f} %)")
else:
    print(f"  [!!] Apprentissage faible : train loss {loss_t[0]:.3f} -> {loss_t[-1]:.3f}")

if loss_e[-1] > min(loss_e) * 1.10:
    print(f"  [!!] Overfitting     : eval loss min={min(loss_e):.3f}, fin={loss_e[-1]:.3f} (remonte)")
else:
    print(f"  [OK] Pas d'overfitting : eval loss reste a son minimum (~{min(loss_e):.3f})")

if abs(gap_f1) < 0.05:
    print(f"  [OK] Generalisation  : F1 val={val_metrics['eval_f1_macro']:.3f} ~ test={test_metrics['eval_f1_macro']:.3f} (ecart {gap_f1:+.3f})")
elif abs(gap_f1) < 0.10:
    print(f"  [~]  Generalisation acceptable : ecart F1 val/test = {gap_f1:+.3f}")
else:
    print(f"  [!!] Generalisation a surveiller : ecart F1 val/test = {gap_f1:+.3f}")

# Cibles projet (cf. docs/e2/STRATEGIE_MODELE_MLOPS.md § 9)
print('\n  Cibles projet : F1 macro >= 0.55, F1 par classe >= 0.20')
ok_macro = test_metrics['eval_f1_macro'] >= 0.55
print(f"    F1 macro test    : {test_metrics['eval_f1_macro']:.3f} {'[OK]' if ok_macro else '[EN-DESSOUS]'}")
for cls in ('neg', 'neu', 'pos'):
    k = f'eval_f1_{cls}'
    if k in test_metrics:
        v = test_metrics[k]
        print(f"    F1 {cls}           : {v:.3f} {'[OK]' if v >= 0.20 else '[EN-DESSOUS]'}")

In [ ]:
# Cellule 7 (optionnelle) — TensorBoard inline pour exploration interactive
# Courbes par step (granularite plus fine que la Cellule 6) :
# train/loss, eval/loss, eval/accuracy, eval/f1_macro, learning_rate
%load_ext tensorboard
%tensorboard --logdir {OUT_DIR}/runs

## Réinjecter le modèle dans le pipeline local

1. télécharger `sentiment_fr_finetuned_colab.zip` depuis Colab
2. en local, dézipper dans `models/sentiment_fr-finetuned-colab/`
   ```powershell
   Expand-Archive -Path .\sentiment_fr_finetuned_colab.zip -DestinationPath .\models\sentiment_fr-finetuned-colab -Force
   ```
3. ouvrir `.env` et remplacer la ligne :
   ```env
   SENTIMENT_FINETUNED_MODEL_PATH=models/sentiment_fr-finetuned-colab
   ```
4. redémarrer l'API (`start_full.bat` ou `python -m src.e2.api.main`)
5. ouvrir le cockpit Streamlit, onglet `IA` → vérifier que `Modèle utilisé` affiche `sentiment_fr-finetuned-colab`
6. tester une prédiction sur une phrase de démo

Le modèle Colab est ainsi devenu le modèle d'inférence en production locale.